実ベクトルにおけるフーリエ回帰

In [1]:
import numpy as np

def estimate_c(x, y, N):
    """
    y_k = sum_{j=-N}^{N} c_j * exp(i * x_k * j)
    の係数 c_j を最小二乗で推定する

    Parameters
    ----------
    x : array-like, shape (M,)
        観測点
    y : array-like, shape (M,)
        観測値（複素数でも実数でも可）
    N : int
        j = -N, ..., N

    Returns
    -------
    j_vals : ndarray, shape (2N+1,)
        j の値
    c_hat : ndarray, shape (2N+1,)
        推定された係数ベクトル
    A : ndarray, shape (M, 2N+1)
        設計行列
    """
    x = np.asarray(x)
    y = np.asarray(y)

    j_vals = np.arange(-N, N + 1)   # [-N, ..., N]

    # 設計行列 A[k, m] = exp(i * x_k * j_vals[m])
    A = np.exp(1j * np.outer(x, j_vals))

    # 最小二乗解
    c_hat, residuals, rank, s = np.linalg.lstsq(A, y, rcond=None)

    return j_vals, c_hat, A

In [2]:
import numpy as np

# 真の係数
N = 2
j_true = np.arange(-N, N + 1)
c_true = np.array([1+0.5j, -0.3+0.2j, 2.0+0j, 0.7-0.1j, -1.2+0.3j])

# 観測点を M >= 2N+1 個用意
M = 20
x = np.linspace(0, 2*np.pi, M, endpoint=False)

# 観測値生成
A = np.exp(1j * np.outer(x, j_true))
y = A @ c_true

# ノイズを少し加える
np.random.seed(0)
noise = 0.01 * (np.random.randn(M) + 1j*np.random.randn(M))
y_noisy = y + noise

# 推定
j_hat, c_hat, A_hat = estimate_c(x, y_noisy, N)

print("j =", j_hat)
print("true c =", c_true)
print("estimated c =", c_hat)

j = [-2 -1  0  1  2]
true c = [ 1. +0.5j -0.3+0.2j  2. +0.j   0.7-0.1j -1.2+0.3j]
estimated c = [ 1.00209755+0.50213996j -0.30033582+0.19984254j  2.00569335+0.0005575j
  0.70208548-0.10138951j -1.20226917+0.29682624j]


楕円関数に拡張したフーリエ回帰

In [3]:
import numpy as np
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, Any

from scipy.special import ellipj, ellipk
from scipy.optimize import minimize_scalar


@dataclass
class EllipticFourierFitResult:
    N: int
    k: float
    j_vals: np.ndarray          # [-N, ..., N]
    coeffs: np.ndarray          # c_j
    y_hat: np.ndarray           # fitted values at training points
    residual_norm: float
    design_matrix: np.ndarray
    info: Dict[str, Any]


def jacobi_basis_matrix(u: np.ndarray, N: int, k: float) -> Tuple[np.ndarray, np.ndarray]:
    """
    A[m, idx] = (cn(u_m, k) + i sn(u_m, k))^j
    where j = -N, ..., N

    Parameters
    ----------
    u : shape (M,)
        sample points
    N : int
        truncation order
    k : float
        elliptic modulus, 0 <= k < 1

    Returns
    -------
    A : shape (M, 2N+1), complex
        design matrix
    j_vals : shape (2N+1,)
        j values
    """
    if not (0.0 <= k < 1.0):
        raise ValueError("k must satisfy 0 <= k < 1.")

    u = np.asarray(u, dtype=float)
    m = k**2

    sn, cn, dn, ph = ellipj(u, m)   # SciPy uses parameter m = k^2
    z = cn + 1j * sn                # z = cn(u,k) + i sn(u,k), |z|=1 for real u

    j_vals = np.arange(-N, N + 1, dtype=int)
    A = z[:, None] ** j_vals[None, :]
    return A, j_vals


def jacobi_dn_weight(u: np.ndarray, k: float) -> np.ndarray:
    """
    Natural weight corresponding to dθ = dn(u,k) du
    Useful if you want the discrete fit to reflect the dn-weighted inner product.
    """
    u = np.asarray(u, dtype=float)
    m = k**2
    sn, cn, dn, ph = ellipj(u, m)
    return np.asarray(dn, dtype=float)


def solve_weighted_least_squares(
    A: np.ndarray,
    y: np.ndarray,
    sample_weights: Optional[np.ndarray] = None,
    ridge: float = 0.0
) -> np.ndarray:
    """
    Solve:
        min_c Σ_m w_m |(A c - y)_m|^2 + ridge * ||c||^2

    If sample_weights is None, ordinary least squares is used.
    """
    y = np.asarray(y)

    if sample_weights is None:
        if ridge <= 0:
            c_hat, *_ = np.linalg.lstsq(A, y, rcond=None)
            return c_hat
        else:
            AH_A = A.conj().T @ A
            AH_y = A.conj().T @ y
            n = A.shape[1]
            c_hat = np.linalg.solve(AH_A + ridge * np.eye(n, dtype=A.dtype), AH_y)
            return c_hat

    w = np.asarray(sample_weights, dtype=float)
    if np.any(w < 0):
        raise ValueError("sample_weights must be nonnegative.")

    sqrt_w = np.sqrt(w)
    Aw = A * sqrt_w[:, None]
    yw = y * sqrt_w

    if ridge <= 0:
        c_hat, *_ = np.linalg.lstsq(Aw, yw, rcond=None)
        return c_hat
    else:
        AH_A = Aw.conj().T @ Aw
        AH_y = Aw.conj().T @ yw
        n = Aw.shape[1]
        c_hat = np.linalg.solve(AH_A + ridge * np.eye(n, dtype=Aw.dtype), AH_y)
        return c_hat


def fit_elliptic_fourier_fixed_k(
    u: np.ndarray,
    y: np.ndarray,
    N: int,
    k: float,
    sample_weights: Optional[np.ndarray] = None,
    use_dn_weight: bool = False,
    ridge: float = 0.0
) -> EllipticFourierFitResult:
    """
    Fit:
        y(u_m) ≈ Σ_{j=-N}^{N} c_j (cn(u_m,k) + i sn(u_m,k))^j

    Parameters
    ----------
    u : shape (M,)
    y : shape (M,)
        real or complex observations
    N : int
    k : float
        fixed elliptic modulus
    sample_weights : optional shape (M,)
        arbitrary discrete weights
    use_dn_weight : bool
        if True, multiply sample_weights by dn(u,k)
    ridge : float
        Tikhonov regularization parameter

    Returns
    -------
    EllipticFourierFitResult
    """
    u = np.asarray(u, dtype=float)
    y = np.asarray(y)

    A, j_vals = jacobi_basis_matrix(u, N, k)

    weights = None
    if sample_weights is not None:
        weights = np.asarray(sample_weights, dtype=float).copy()

    if use_dn_weight:
        dn_w = jacobi_dn_weight(u, k)
        weights = dn_w if weights is None else weights * dn_w

    coeffs = solve_weighted_least_squares(A, y, sample_weights=weights, ridge=ridge)
    y_hat = A @ coeffs

    if weights is None:
        residual_norm = float(np.linalg.norm(y_hat - y))
    else:
        residual_norm = float(np.sqrt(np.sum(weights * np.abs(y_hat - y)**2)))

    return EllipticFourierFitResult(
        N=N,
        k=float(k),
        j_vals=j_vals,
        coeffs=coeffs,
        y_hat=y_hat,
        residual_norm=residual_norm,
        design_matrix=A,
        info={
            "weighted": weights is not None,
            "use_dn_weight": use_dn_weight,
            "ridge": ridge,
            "period_u": 4.0 * ellipk(k**2),
        }
    )


def predict_elliptic_fourier(
    u_new: np.ndarray,
    coeffs: np.ndarray,
    N: int,
    k: float
) -> np.ndarray:
    """
    Predict at new points:
        y(u) = Σ c_j (cn(u,k) + i sn(u,k))^j
    """
    A_new, j_vals = jacobi_basis_matrix(np.asarray(u_new, dtype=float), N, k)
    coeffs = np.asarray(coeffs)
    if coeffs.shape[0] != len(j_vals):
        raise ValueError("coeffs length must be 2N+1.")
    return A_new @ coeffs


def _objective_for_k(
    k: float,
    u: np.ndarray,
    y: np.ndarray,
    N: int,
    sample_weights: Optional[np.ndarray],
    use_dn_weight: bool,
    ridge: float
) -> float:
    """
    Residual objective as a function of k.
    """
    result = fit_elliptic_fourier_fixed_k(
        u=u,
        y=y,
        N=N,
        k=k,
        sample_weights=sample_weights,
        use_dn_weight=use_dn_weight,
        ridge=ridge
    )
    return result.residual_norm


def fit_elliptic_fourier(
    u: np.ndarray,
    y: np.ndarray,
    N: int,
    k: Optional[float] = None,
    k_bounds: Tuple[float, float] = (0.0, 0.999),
    k_grid_size: int = 81,
    refine_k: bool = True,
    sample_weights: Optional[np.ndarray] = None,
    use_dn_weight: bool = False,
    ridge: float = 0.0
) -> EllipticFourierFitResult:
    """
    Main interface.

    If k is given:
        fit c_j only.

    If k is None:
        estimate k by
        1) grid search on k
        2) optional bounded scalar refinement

    Notes
    -----
    - k should stay away from 1 for numerical stability.
    - when k=0, this reduces to ordinary Fourier fitting:
          cn(u,0)=cos(u), sn(u,0)=sin(u)
      so basis becomes exp(i j u).
    """
    u = np.asarray(u, dtype=float)
    y = np.asarray(y)

    if u.ndim != 1 or y.ndim != 1:
        raise ValueError("u and y must be 1D arrays.")
    if len(u) != len(y):
        raise ValueError("u and y must have the same length.")
    if len(u) < 2 * N + 1:
        raise ValueError("Need at least 2N+1 sample points.")

    if k is not None:
        return fit_elliptic_fourier_fixed_k(
            u=u,
            y=y,
            N=N,
            k=k,
            sample_weights=sample_weights,
            use_dn_weight=use_dn_weight,
            ridge=ridge
        )

    k_min, k_max = k_bounds
    if not (0.0 <= k_min < k_max < 1.0):
        raise ValueError("Require 0 <= k_min < k_max < 1.")

    # --- coarse grid search ---
    k_grid = np.linspace(k_min, k_max, k_grid_size)
    obj_vals = np.array([
        _objective_for_k(
            k=kg,
            u=u,
            y=y,
            N=N,
            sample_weights=sample_weights,
            use_dn_weight=use_dn_weight,
            ridge=ridge
        )
        for kg in k_grid
    ])

    best_idx = int(np.argmin(obj_vals))
    best_k = float(k_grid[best_idx])
    best_obj = float(obj_vals[best_idx])

    # --- optional refinement ---
    refine_info = None
    if refine_k:
        # local bracket around best grid point
        left_idx = max(best_idx - 1, 0)
        right_idx = min(best_idx + 1, len(k_grid) - 1)
        a = float(k_grid[left_idx])
        b = float(k_grid[right_idx])

        # if the best point is at an edge, use full bounds
        if a == b:
            a, b = k_min, k_max
        elif a == best_k or b == best_k:
            a, b = k_min, k_max

        opt = minimize_scalar(
            _objective_for_k,
            bounds=(a, b),
            method="bounded",
            args=(u, y, N, sample_weights, use_dn_weight, ridge)
        )
        if opt.success and opt.fun <= best_obj:
            best_k = float(opt.x)
            best_obj = float(opt.fun)
        refine_info = {
            "success": bool(opt.success),
            "message": str(opt.message),
            "x": float(opt.x),
            "fun": float(opt.fun),
            "interval": (a, b),
        }

    result = fit_elliptic_fourier_fixed_k(
        u=u,
        y=y,
        N=N,
        k=best_k,
        sample_weights=sample_weights,
        use_dn_weight=use_dn_weight,
        ridge=ridge
    )

    result.info.update({
        "k_estimated": True,
        "k_grid_size": k_grid_size,
        "k_bounds": k_bounds,
        "grid_best_k": float(k_grid[best_idx]),
        "grid_best_obj": float(obj_vals[best_idx]),
        "refine_info": refine_info,
    })
    return result

In [4]:
import numpy as np

# 例: 真のパラメータ
N = 3
k_true = 0.7
j_vals = np.arange(-N, N + 1)

# 真の係数（複素）
c_true = np.array([
    0.2 - 0.1j,
    -0.4 + 0.3j,
    0.7 + 0.0j,
    1.2 + 0.0j,
    0.1 - 0.5j,
    -0.2 + 0.2j,
    0.05 + 0.1j
], dtype=complex)

# サンプル点
M = 200
period_u = 4.0 * ellipk(k_true**2)
u = np.linspace(0, period_u, M, endpoint=False)

# 観測値生成
A_true, _ = jacobi_basis_matrix(u, N, k_true)
y_clean = A_true @ c_true

# ノイズ
rng = np.random.default_rng(0)
noise = 0.03 * (rng.standard_normal(M) + 1j * rng.standard_normal(M))
y_obs = y_clean + noise

# フィット
result = fit_elliptic_fourier_fixed_k(
    u=u,
    y=y_obs,
    N=N,
    k=k_true,
    use_dn_weight=False,   # 必要なら True
    ridge=1e-8
)

print("推定 k =", result.k)
print("j =", result.j_vals)
print("真の係数 =", c_true)
print("推定係数 =", result.coeffs)
print("残差ノルム =", result.residual_norm)

推定 k = 0.7
j = [-3 -2 -1  0  1  2  3]
真の係数 = [ 0.2 -0.1j -0.4 +0.3j  0.7 +0.j   1.2 +0.j   0.1 -0.5j -0.2 +0.2j
  0.05+0.1j]
推定係数 = [ 0.20001159-0.10135299j -0.40257026+0.29859023j  0.6999262 +0.00289146j
  1.20017078-0.00284385j  0.09802665-0.50048393j -0.20086095+0.19915398j
  0.05057954+0.10183419j]
残差ノルム = 0.5913042245607827


In [5]:
result2 = fit_elliptic_fourier(
    u=u,
    y=y_obs,
    N=N,
    k=None,                    # ← k を未知として推定
    k_bounds=(0.0, 0.95),
    k_grid_size=101,
    refine_k=True,
    use_dn_weight=False,
    ridge=1e-8
)

print("真の k =", k_true)
print("推定 k =", result2.k)
print("推定係数 =", result2.coeffs)
print("残差ノルム =", result2.residual_norm)
print("info =", result2.info)

真の k = 0.7
推定 k = 0.7058832624591773
推定係数 = [ 0.196842  -0.1111591j  -0.39585027+0.3069684j   0.69892312-0.0052617j
  1.20068715-0.0010201j   0.10424593-0.49668172j -0.20468613+0.19778937j
  0.04907385+0.10432069j]
残差ノルム = 0.5897772959394502
info = {'weighted': False, 'use_dn_weight': False, 'ridge': 1e-08, 'period_u': np.float64(7.410451051035845), 'k_estimated': True, 'k_grid_size': 101, 'k_bounds': (0.0, 0.95), 'grid_best_k': 0.703, 'grid_best_obj': 0.590144721574406, 'refine_info': {'success': True, 'message': 'Solution found.', 'x': 0.7058832624591773, 'fun': 0.5897772959394502, 'interval': (0.6935, 0.7125)}}


In [6]:
u_new = np.linspace(0, period_u, 500, endpoint=False)
y_pred = predict_elliptic_fourier(u_new, result2.coeffs, N=N, k=result2.k)